# WebDoMiner Colab Demo

## Open-Web Corpus Generation from Requirements Specification Documents

This notebook demonstrates **WebDoMiner**, a Python project that generates a domain-specific corpus from the open web using a natural-language **Requirements Specification (RS)** document.

### What WebDoMiner does
WebDoMiner takes an RS document as input and performs the following steps:

1. Extracts keywords from the RS
2. Builds search queries
3. Discovers relevant public URLs
4. Scrapes and cleans page content
5. Applies semantic filtering
6. Saves a structured JSONL corpus

### Demo goal
This Colab notebook provides a simple demonstration of the full project workflow using the example RS included in the repository.

> Note: For Colab simplicity and reliability, this demo disables the Playwright fallback by default and focuses on the simplest working path.



In [1]:
REPO_URL = "https://github.com/SoftALL/WebDoMiner-3.0.git"

!git clone "$REPO_URL"
%cd /content/WebDoMiner-3.0

!pip install -q -U pip
!pip install -q -e .

Cloning into 'WebDoMiner-3.0'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 149 (delta 64), reused 118 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 92.49 KiB | 3.70 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/WebDoMiner-3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for webdominer (pyproject.toml) ... done


## Imports and helper functions

This section imports helper utilities used to preview the generated output files inside Colab.

In [2]:
import json
from pathlib import Path
import pandas as pd

def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def preview_text(text, limit=300):
    text = str(text)
    return text if len(text) <= limit else text[:limit] + "..."

def display_df(rows, columns=None, text_columns=None, max_rows=10):
    if not rows:
        print("No rows to display.")
        return
    df = pd.DataFrame(rows)
    if columns:
        existing = [c for c in columns if c in df.columns]
        df = df[existing]
    if text_columns:
        for col in text_columns:
            if col in df.columns:
                df[col] = df[col].apply(preview_text)
    display(df.head(max_rows))

## Input RS document

For this demo, we use the sample RS document already included in the repository:

`examples/sample_rs.txt`

This keeps the notebook self-contained and easy to reproduce.

In [3]:
sample_rs_path = Path("examples/sample_rs.txt")

print("Using input file:", sample_rs_path)
print()

with open(sample_rs_path, "r", encoding="utf-8") as f:
    rs_text = f.read()

print(rs_text[:2000])

Using input file: examples/sample_rs.txt

FleetOpt Requirements Specification

FleetOpt is a web-based logistics operations platform intended for regional delivery companies. The goal of the system is to streamline order scheduling, vehicle assignment, route planning, driver availability management, package tracking, and billing coordination. The system shall support dispatchers, drivers, operations managers, and administrators while preserving security, auditability, and operational efficiency.

The platform shall provide an order scheduling module that allows dispatch staff to create, update, cancel, and reschedule delivery jobs. The scheduling workflow shall consider package priority, delivery windows, vehicle capacity, driver shift availability, and depot operating hours. The system shall prevent conflicting assignments and shall warn staff when jobs overlap with blocked time, unavailable vehicles, or unavailable drivers.

The vehicle and driver management component shall allow adm

## Run the full pipeline

This is the main demonstration cell.

It runs the complete WebDoMiner pipeline end to end using the packaged command-line interface.
For Colab simplicity, the run uses:
- a small number of keywords
- a small number of URLs per keyword
- semantic filtering enabled
- Playwright fallback disabled

In [4]:
!python -m webdominer.cli \
  --input "examples/sample_rs.txt" \
  --top-keywords 5 \
  --top-urls 3 \
  --similarity-threshold 0.45 \
  --min-word-count 120 \
  --disable-playwright-fallback \
  --log-level INFO

2026-04-09 17:23:43 | INFO | webdominer.pipeline | Starting WebDoMiner pipeline.
2026-04-09 17:23:43 | INFO | webdominer.pipeline | Loading RS document from: examples/sample_rs.txt
modules.json: 100% 349/349 [00:00<00:00, 1.18MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 284kB/s]
README.md: 10.5kB [00:00, 28.6MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 245kB/s]
2026-04-09 17:23:44 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
config.json: 100% 612/612 [00:00<00:00, 2.96MB/s]
model.safetensors: 100% 90.9M/90.9M [00:00<00:00, 109MB/s]
Loading weights: 100% 103/103 [00:00<00:00, 1701.36it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED

## Locate generated output files

After the full pipeline run, this cell checks which output files were produced.

In [5]:
from pathlib import Path

possible_output_dirs = [
    Path("data/output"),
    Path("data"),
    Path("."),
]

for d in possible_output_dirs:
    print(f"\nChecking directory: {d.resolve()}")
    if d.exists():
        for p in sorted(d.glob("*")):
            print(" -", p)
    else:
        print(" Directory does not exist.")


Checking directory: /content/WebDoMiner-3.0/data/output
 - data/output/corpus.jsonl
 - data/output/failed.jsonl
 - data/output/rejected.jsonl
 - data/output/summary.json

Checking directory: /content/WebDoMiner-3.0/data
 - data/input
 - data/output

Checking directory: /content/WebDoMiner-3.0
 - .git
 - .gitignore
 - LICENSE
 - README.md
 - data
 - examples
 - logs
 - pyproject.toml
 - tests
 - webdominer
 - webdominer.egg-info


## Load the generated files

This section loads the JSONL and summary outputs so they can be inspected directly in the notebook.

In [6]:
from pathlib import Path

accepted_output_file = Path("data/output/corpus.jsonl")
rejected_output_file = Path("data/output/rejected.jsonl")
failed_output_file = Path("data/output/failed.jsonl")
summary_output_file = Path("data/output/summary.json")

print("Accepted exists:", accepted_output_file.exists(), accepted_output_file)
print("Rejected exists:", rejected_output_file.exists(), rejected_output_file)
print("Failed exists:  ", failed_output_file.exists(), failed_output_file)
print("Summary exists: ", summary_output_file.exists(), summary_output_file)

Accepted exists: True data/output/corpus.jsonl
Rejected exists: True data/output/rejected.jsonl
Failed exists:   True data/output/failed.jsonl
Summary exists:  True data/output/summary.json


## Preview accepted corpus output

These are the pages that passed scraping and semantic filtering and were accepted into the final corpus.

In [7]:
accepted_jsonl = load_jsonl(accepted_output_file)

display_df(
    accepted_jsonl,
    columns=["matched_keyword", "title", "source_url", "similarity_score", "extraction_method", "text"],
    text_columns=["title", "source_url", "text"],
    max_rows=5
)

,matched_keyword,title,source_url,similarity_score,extraction_method,text
0,vehicle capacity driver,Driver management software,https://www.chevinfleet.com/en-us/driver-manag...,0.6675,trafilatura,Keep your drivers up to speed\nWe know that yo...
1,specification fleetopt web,Fleet management - Wikipedia,https://en.wikipedia.org/wiki/Fleet_management,0.6578,trafilatura,Fleet management is the management of:\n- Comm...
2,vehicle capacity driver,Route Planning & Optimization Software for Fle...,https://fareye.com/products/route-planning-sof...,0.6321,trafilatura,Route Planning Software to Reduce Delivery Cos...
3,vehicle capacity maintenance,Fleet Optimization Software Solutions,https://www.flexfulfillment.eu/top-8-fleet-opt...,0.6296,trafilatura,The Shipping Speed Trap: How to Deliver in Day...
4,vehicle capacity driver,LogiFleet,https://leafkutter.com/fleet-management-portal,0.6267,trafilatura,"Smart Fleet Management, Real-Time Control.\nTr..."


## Preview rejected pages

These are pages that were discovered or scraped but later rejected, for example due to low content quality or low semantic similarity.

In [8]:
rejected_jsonl = load_jsonl(rejected_output_file)

display_df(
    rejected_jsonl,
    columns=["url", "reason", "matched_keyword", "title", "similarity_score"],
    text_columns=["url", "reason", "title"],
    max_rows=5
)

,url,reason,matched_keyword,title,similarity_score
0,https://arxiv.org/abs/2603.16514,below_similarity_threshold:0.3931,specification fleetopt web,FleetOpt: Analytical Fleet Provisioning for LL...,0.3931
1,https://en.wikipedia.org/wiki/Device_driver,below_similarity_threshold:0.3022,drivers operations managers,Device driver - Wikipedia,0.3022
2,https://en.wikipedia.org/wiki/Operations_manag...,below_similarity_threshold:0.3957,drivers operations managers,Operations management - Wikipedia,0.3957
3,https://fareye.com/resources/blogs/vehicle-rou...,low_value_or_blocked_page,vehicle capacity driver,Why Vehicle Routing Software is Essential for ...,NaN
4,https://financeandjobs.com/uber-is-looking-for...,below_similarity_threshold:0.3107,drivers operations managers,Uber is looking for people to work as delivery...,0.3107


## Preview failed pages

These are pages that could not be processed successfully, for example due to request failures, extraction issues, or blocked access.

In [9]:
failed_jsonl = load_jsonl(failed_output_file)

display_df(
    failed_jsonl,
    columns=["url", "error", "matched_keyword", "query"],
    text_columns=["url", "error", "query"],
    max_rows=5
)

,url,error,matched_keyword,query
0,https://fleetopt.io/,ConnectTimeout: HTTPSConnectionPool(host='flee...,specification fleetopt web,specification fleetopt web workflow system
1,https://fleetopt.io/about,ConnectTimeout: HTTPSConnectionPool(host='flee...,fleetopt web,fleetopt web software system
2,https://fleetopt.io/solutions,ConnectTimeout: HTTPSConnectionPool(host='flee...,fleetopt web,fleetopt web software system
3,https://www.fleetio.com/blog/the-best-preventi...,HTTPError: 403 Client Error: Forbidden for url...,vehicle capacity maintenance,"""vehicle capacity maintenance"""
4,https://www.fleetio.com/integrations-directory...,HTTPError: 403 Client Error: Forbidden for url...,specification fleetopt web,"""specification fleetopt web"""


## Show summary output

This summary file provides a compact overview of the pipeline run and the number of accepted, rejected, and failed pages.

In [10]:
if summary_output_file.exists():
    with open(summary_output_file, "r", encoding="utf-8") as f:
        summary_json = json.load(f)
    print(json.dumps(summary_json, indent=2, ensure_ascii=False))
else:
    print("Summary file not found.")

{
  "summary": {
    "keywords_extracted": 5,
    "raw_search_results": 45,
    "unique_urls_discovered": 38,
    "pages_scraped_successfully": 20,
    "pages_rejected": 19,
    "pages_failed": 6,
    "final_accepted_documents": 13,
    "started_at": "2026-04-09T17:23:43.666813+00:00",
    "finished_at": "2026-04-09T17:26:22.109621+00:00"
  },
  "input_file": "examples/sample_rs.txt",
  "accepted_output_file": "/content/WebDoMiner-3.0/data/output/corpus.jsonl",
  "rejected_output_file": "/content/WebDoMiner-3.0/data/output/rejected.jsonl",
  "failed_output_file": "/content/WebDoMiner-3.0/data/output/failed.jsonl",
  "keywords": [
    {
      "phrase": "vehicle capacity driver",
      "score": 0.5587,
      "source": "keybert",
      "token_count": 3
    },
    {
      "phrase": "drivers operations managers",
      "score": 0.5564,
      "source": "keybert",
      "token_count": 3
    },
    {
      "phrase": "vehicle capacity maintenance",
      "score": 0.5456,
      "source": "keyber

## Notes

- This notebook demonstrates the full WebDoMiner workflow in a simple Colab-friendly format.
- It uses the packaged CLI so that the reviewer can see the project running as an end-to-end system.
- For simplicity, the Playwright fallback is disabled in this demo.
- The repository also contains a modular internal implementation for keyword extraction, discovery, scraping, semantic filtering, and structured output writing.